# 加州房价预测：经典机器学习教程

[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/greathousesh/transformer-tutorial/blob/main/california_housing_tutorial.ipynb)

本 notebook 通过 **California Housing**（加州房价）数据集，
完整演示一个**端到端的回归任务**：从数据加载、探索性分析（EDA）、特征工程，
到训练多种模型并进行对比、调优、可解释性分析；最后再用 PyTorch MLP 与 GBR 同台对比。

## 为什么选 California Housing
- 是 **scikit-learn 官方推荐的 Boston Housing 替代品**（自 sklearn 1.2 起 Boston 已被移除）；
- **无伦理争议**，数据来自 1990 年加州人口普查；
- **20,640 行 × 8 特征**——比 Boston 大 40 倍，足以让深度学习与梯度提升公平较量；
- **自带经纬度**，可以做地理可视化，比纯结构化的 Boston 多一个维度；
- `sklearn.datasets.fetch_california_housing` 一键加载，Kaggle 默认环境无需额外下载。

## 目标读者
- 刚接触机器学习、想要走完一个完整流程的同学
- 想复习线性回归 / 正则化 / 树模型 / 集成学习的工程师
- 想了解残差分析、特征重要性、地理可视化等模型诊断手段的人

## 目录
1. 数据集介绍与加载
2. 探索性数据分析（EDA）
3. 特征关系与地理可视化
4. 数据预处理（划分、标准化）
5. 基线模型：线性回归
6. 正则化：Ridge & Lasso
7. 树模型：决策树、随机森林、梯度提升
8. 交叉验证与超参搜索
9. 模型对比汇总
10. 特征重要性分析
11. 残差诊断
12. 深度学习对照：PyTorch MLP
13. 总结与延伸


## 1. 环境与依赖

本教程仅依赖标准的 PyData + scikit-learn + PyTorch 工具链，Kaggle 默认环境已全部预装。


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

print(f'numpy   : {np.__version__}')
print(f'pandas  : {pd.__version__}')
import sklearn
print(f'sklearn : {sklearn.__version__}')


## 2. 数据集介绍与加载

**California Housing** 数据集来自 1990 年美国人口普查，每行代表一个**人口普查街区组**
（block group，通常 600~3000 人）。目标变量 `MedHouseVal` 是该街区组的**房价中位数**，
单位为 **\$100,000**（即 2.5 表示 \$250,000）。

| 特征 | 含义 |
| --- | --- |
| MedInc | 街区组家庭收入中位数（单位：万美元） |
| HouseAge | 房屋年龄中位数（年） |
| AveRooms | 平均房间数（每户） |
| AveBedrms | 平均卧室数（每户） |
| Population | 街区组总人口 |
| AveOccup | 平均每户人数 |
| Latitude | 纬度 |
| Longitude | 经度 |
| **MedHouseVal** | **房价中位数（目标，单位 \$100K）** |

> 📍 **注意**：`MedHouseVal` 在 5.0001 处被**截断**（即 \$500,001 封顶），与 Boston 的 \$50K 截断
> 是同一类问题。残差分析时我们会观察到这个现象。


In [ ]:
bunch = fetch_california_housing(as_frame=True)
df = bunch.frame
print(f'数据形状: {df.shape}')
print(f'特征列: {bunch.feature_names}')
print(f'目标列: {bunch.target_names[0]}')
df.head()


## 3. 探索性数据分析（EDA）

EDA 是建模前必做的一步：了解数据的形状、量纲、缺失情况、异常值与目标变量的分布。


In [ ]:
print('数据类型与非空数量:')
df.info()

print('\n缺失值统计:')
print(df.isnull().sum().to_string())

print('\n描述性统计:')
df.describe().round(3)


### 3.1 目标变量分布

观察 `MedHouseVal` 的分布：是否对称？是否需要做 log 变换？是否有截断（封顶）？


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['MedHouseVal'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('MedHouseVal Distribution')
axes[0].set_xlabel('MedHouseVal (in $100K)')
axes[0].set_ylabel('Count')

axes[1].boxplot(df['MedHouseVal'], vert=False)
axes[1].set_title('MedHouseVal Boxplot')
axes[1].set_xlabel('MedHouseVal (in $100K)')

plt.tight_layout()
plt.show()

cap_count = (df['MedHouseVal'] >= 5.0).sum()
print(f"封顶值 (>=5.0, 即 $500K) 的记录数: {cap_count} ({cap_count / len(df) * 100:.1f}%)")
print(f"偏度 (skewness):  {df['MedHouseVal'].skew():.3f}")
print(f"峰度 (kurtosis): {df['MedHouseVal'].kurt():.3f}")


**观察**：
- `MedHouseVal` 分布右偏（skewness > 0），尾部有少量高价区；
- 在 5.0 处出现明显堆积，~5% 的样本被**截断在 \$500,000**——这是右截断问题，
  会让模型对高价区的预测系统性低估，残差分析时会清晰看到。


### 3.2 特征分布

快速浏览所有特征的分布，识别长尾、异常值。


In [ ]:
features = [c for c in df.columns if c != 'MedHouseVal']
fig, axes = plt.subplots(2, 4, figsize=(15, 6))
for ax, col in zip(axes.flat, features):
    ax.hist(df[col], bins=40, color='steelblue', edgecolor='white')
    ax.set_title(col)
plt.tight_layout()
plt.show()


**观察**：
- `MedInc`、`AveRooms`、`AveBedrms`、`Population`、`AveOccup` 都明显右偏，存在长尾;
- `HouseAge` 在 52 处也有截断（许多老房子被归入「52 年及以上」）;
- `Latitude` / `Longitude` 显示双峰——对应**北加州（旧金山湾区）**与**南加州（洛杉矶）**两大都市群。


## 4. 特征关系与地理可视化

### 4.1 相关性热图


In [ ]:
corr = df.corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Heatmap', fontsize=13)
plt.tight_layout()
plt.show()

print('与 MedHouseVal 相关性最强的 5 个特征:')
print(corr['MedHouseVal'].drop('MedHouseVal').abs().sort_values(ascending=False).head(5))


**关键发现**:
- `MedInc`(收入中位数)与 `MedHouseVal` 强正相关(≈ +0.69),压倒性最重要的特征;
- `AveRooms` / `AveBedrms` 高度共线(≈ 0.85),建模时要注意;
- `Latitude` / `Longitude` 单独看相关性不强,但**地理位置**通过它们的非线性组合影响房价。


### 4.2 地理可视化:加州地图上的房价

把每个街区组按经纬度画在地图上,颜色编码房价,会得到一张**加州房价地图**——
旧金山湾区与洛杉矶/圣地亚哥沿海是两个明显的高价区。


In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sc = ax.scatter(df['Longitude'], df['Latitude'],
                c=df['MedHouseVal'], cmap='viridis',
                s=df['Population'] / 200, alpha=0.4, edgecolors='none')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('California House Prices by Location\n(point size ∝ population, color = MedHouseVal)')
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('MedHouseVal ($100K)')
plt.tight_layout()
plt.show()


**直观结论**:
- **沿海高价**:旧金山湾区(北纬 37–38)与洛杉矶/圣地亚哥(北纬 33–34)沿海明显偏黄;
- **内陆低价**:中央谷地与内陆山区几乎全是紫色;
- **经纬度本身不线性,但树模型可以自动学到「沿海 vs 内陆」「南加州 vs 北加州」的分割**。
这是树模型在这份数据上常常胜出线性模型的关键原因。


### 4.3 顶部相关特征 vs 房价


In [ ]:
top_features = ['MedInc', 'AveRooms', 'HouseAge', 'AveOccup', 'Latitude', 'Longitude']
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for ax, feat in zip(axes.flat, top_features):
    ax.scatter(df[feat], df['MedHouseVal'], alpha=0.15, s=8, color='steelblue')
    ax.set_xlabel(feat)
    ax.set_ylabel('MedHouseVal')
    ax.set_title(f'{feat} vs MedHouseVal')

plt.tight_layout()
plt.show()


**观察**:
- `MedInc` 与目标几乎线性,但顶部 5.0 的横线就是封顶截断;
- `AveRooms`、`AveOccup` 有极端长尾(出现 100+ 间房或 1000+ 占用),实际是数据噪声;
- `Latitude` / `Longitude` 单变量看不出关系,但联合起来就是地图(见 4.2)。


## 5. 数据预处理

### 5.1 训练 / 测试划分


In [ ]:
X = df.drop(columns='MedHouseVal').values
y = df['MedHouseVal'].values
feature_names = df.drop(columns='MedHouseVal').columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f'训练集: X={X_train.shape}, y={y_train.shape}')
print(f'测试集: X={X_test.shape},  y={y_test.shape}')


### 5.2 标准化

线性模型 / 神经网络对量纲敏感,使用 `StandardScaler`;树模型对量纲不敏感,直接用原始数据。

> **重要**:`scaler.fit` 只在训练集上做,再 `transform` 测试集,避免数据泄漏。


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('标准化后训练集前 3 行:')
pd.DataFrame(X_train_scaled, columns=feature_names).head(3).round(3)


### 5.3 评估辅助函数

定义统一的评估器,输出 RMSE / MAE / R²,并支持 5 折交叉验证。


In [ ]:
def evaluate(model, X_tr, y_tr, X_te, y_te, name, do_cv=True):
    model.fit(X_tr, y_tr)
    pred_tr = model.predict(X_tr)
    pred_te = model.predict(X_te)

    if do_cv:
        cv = cross_val_score(model, X_tr, y_tr, cv=5,
                             scoring='neg_root_mean_squared_error', n_jobs=-1)
        cv_rmse = -cv.mean()
        cv_std = cv.std()
    else:
        cv_rmse, cv_std = float('nan'), float('nan')

    return {
        'model': name,
        'train_rmse': np.sqrt(mean_squared_error(y_tr, pred_tr)),
        'test_rmse': np.sqrt(mean_squared_error(y_te, pred_te)),
        'test_mae': mean_absolute_error(y_te, pred_te),
        'test_r2': r2_score(y_te, pred_te),
        'cv_rmse': cv_rmse,
        'cv_std': cv_std,
        'fitted': model,
    }


def print_result(r):
    cv_str = f"{r['cv_rmse']:.3f} ± {r['cv_std']:.3f}" if not np.isnan(r['cv_rmse']) else 'n/a'
    print(f"{r['model']:<22} | "
          f"Train RMSE: {r['train_rmse']:.3f} | "
          f"Test RMSE: {r['test_rmse']:.3f} | "
          f"Test MAE: {r['test_mae']:.3f} | "
          f"Test R²: {r['test_r2']:.3f} | "
          f"CV RMSE: {cv_str}")


results = []


## 6. 基线模型:线性回归

线性回归 ($\hat y = w^\top x + b$) 是最经典的基线,便于解释,
也能让我们看到「不做任何花哨的事,模型能到什么水平」。


In [ ]:
r = evaluate(LinearRegression(), X_train_scaled, y_train,
             X_test_scaled, y_test, 'Linear Regression')
print_result(r)
results.append(r)

coef = pd.Series(r['fitted'].coef_, index=feature_names).sort_values()
plt.figure(figsize=(8, 5))
coef.plot(kind='barh', color=['#d62728' if v < 0 else '#2ca02c' for v in coef])
plt.title('Linear Regression Coefficients (on standardized features)')
plt.xlabel('Coefficient')
plt.axvline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()


**解读系数**(特征已标准化,可直接比较绝对值大小):
- `MedInc` 系数最正 → 收入越高,房价越高;
- `Latitude` / `Longitude` 系数显著为负 → 越往北、越往东,房价越低
  (但这是**线性近似**,真实关系是非线性的——见 §4.2 地图);
- `AveBedrms` 系数为正,而 `AveRooms` 为负——这是它们高度共线导致的「系数翻转」,
  正则化模型会缓解这个问题。


## 7. 正则化:Ridge 与 Lasso

- **Ridge (L2)**:$\min \|y - Xw\|^2 + \alpha \|w\|_2^2$,缩小所有系数;
- **Lasso (L1)**:$\min \|y - Xw\|^2 + \alpha \|w\|_1$,把不重要特征的系数压到 0(自动特征选择)。

我们用 `GridSearchCV` 在训练集上选出最佳 $\alpha$。


In [ ]:
ridge_grid = GridSearchCV(
    Ridge(),
    param_grid={'alpha': np.logspace(-3, 3, 25)},
    cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
ridge_grid.fit(X_train_scaled, y_train)
print(f'Ridge 最佳 alpha: {ridge_grid.best_params_["alpha"]:.4f}')

lasso_grid = GridSearchCV(
    Lasso(max_iter=20000),
    param_grid={'alpha': np.logspace(-4, 0, 20)},
    cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
lasso_grid.fit(X_train_scaled, y_train)
print(f'Lasso 最佳 alpha: {lasso_grid.best_params_["alpha"]:.4f}')

r_ridge = evaluate(Ridge(alpha=ridge_grid.best_params_['alpha']),
                   X_train_scaled, y_train, X_test_scaled, y_test, 'Ridge')
r_lasso = evaluate(Lasso(alpha=lasso_grid.best_params_['alpha'], max_iter=20000),
                   X_train_scaled, y_train, X_test_scaled, y_test, 'Lasso')
print_result(r_ridge)
print_result(r_lasso)
results.extend([r_ridge, r_lasso])


In [ ]:
coef_df = pd.DataFrame({
    'Linear': results[0]['fitted'].coef_,
    'Ridge':  r_ridge['fitted'].coef_,
    'Lasso':  r_lasso['fitted'].coef_,
}, index=feature_names).sort_values('Linear')

coef_df.plot(kind='barh', figsize=(9, 5), width=0.8)
plt.title('Coefficient Comparison: Linear vs Ridge vs Lasso')
plt.axvline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

zeroed = coef_df.index[(coef_df['Lasso'] == 0)].tolist()
print(f'Lasso 压到 0 的特征: {zeroed}')


## 8. 树模型:决策树、随机森林、梯度提升

树模型直接处理原始特征,能自然捕捉非线性、特征间的交互(比如经纬度组合),
且不需要标准化。我们使用**未标准化**的 `X_train` / `X_test`。

> ⏱ 在 20K 行数据上,随机森林与 GBR 的训练比 Boston 慢得多,但仍然在分钟级以内。


In [ ]:
r_dt = evaluate(DecisionTreeRegressor(max_depth=8, random_state=42),
                X_train, y_train, X_test, y_test, 'Decision Tree (d=8)')
print_result(r_dt)

r_rf = evaluate(RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42),
                X_train, y_train, X_test, y_test, 'Random Forest')
print_result(r_rf)

r_gb = evaluate(GradientBoostingRegressor(n_estimators=200,
                                          learning_rate=0.1,
                                          max_depth=3,
                                          random_state=42),
                X_train, y_train, X_test, y_test, 'Gradient Boosting')
print_result(r_gb)
results.extend([r_dt, r_rf, r_gb])


**观察**:
- 单棵决策树训练 RMSE 远小于测试 RMSE → 过拟合;
- 随机森林通过 bagging 显著降方差,泛化更稳;
- 梯度提升通常拿到最好的成绩——它逐棵拟合上一轮的残差,是 Kaggle 表格类竞赛的常胜将军。


### 8.1 对梯度提升做超参搜索

为控制运行时间,这里用一个**精简网格**(8 个组合):


In [ ]:
gb_grid = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    param_grid={
        'n_estimators': [200, 400],
        'learning_rate': [0.05, 0.1],
        'max_depth': [3, 4],
    },
    cv=3, scoring='neg_root_mean_squared_error', n_jobs=-1)
gb_grid.fit(X_train, y_train)

print(f'GBR 最佳参数: {gb_grid.best_params_}')
print(f'GBR 最佳 CV RMSE: {-gb_grid.best_score_:.3f}')

r_gb_tuned = evaluate(GradientBoostingRegressor(**gb_grid.best_params_, random_state=42),
                      X_train, y_train, X_test, y_test, 'GBR (tuned)', do_cv=False)
r_gb_tuned['cv_rmse'] = -gb_grid.best_score_
print_result(r_gb_tuned)
results.append(r_gb_tuned)


## 9. 模型对比汇总

把所有模型按测试集 RMSE 排序,一眼看清谁更强。


In [ ]:
leaderboard = pd.DataFrame([
    {'Model': r['model'],
     'Train RMSE': r['train_rmse'],
     'Test RMSE':  r['test_rmse'],
     'Test MAE':   r['test_mae'],
     'Test R²':    r['test_r2'],
     'CV RMSE':    r['cv_rmse']}
    for r in results
]).sort_values('Test RMSE').reset_index(drop=True)

leaderboard.round(3)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(leaderboard))
ax.bar(x - 0.2, leaderboard['Train RMSE'], 0.4, label='Train RMSE', color='lightsteelblue')
ax.bar(x + 0.2, leaderboard['Test RMSE'],  0.4, label='Test RMSE',  color='steelblue')
ax.set_xticks(x)
ax.set_xticklabels(leaderboard['Model'], rotation=30, ha='right')
ax.set_ylabel('RMSE')
ax.set_title('Train vs Test RMSE per Model')
ax.legend()
plt.tight_layout()
plt.show()


**结论**:
- 集成树模型(RF、GBR)在 Test RMSE 上明显优于线性族;
- 训练 RMSE 远小于测试 RMSE 通常意味着过拟合——单棵决策树最为典型;
- 调过参的 GBR 在交叉验证与测试集上都最稳。


## 10. 特征重要性

树模型自带 `feature_importances_`,可以告诉我们模型最依赖哪些特征。


In [ ]:
best_tree_model = r_gb_tuned['fitted']
imp = pd.Series(best_tree_model.feature_importances_,
                index=feature_names).sort_values()

plt.figure(figsize=(8, 4))
imp.plot(kind='barh', color='seagreen')
plt.title('Feature Importance — Tuned Gradient Boosting')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print('Top 5 重要特征:')
print(imp.sort_values(ascending=False).head(5))


**与 EDA 的相关性结论一致**:`MedInc`(收入)是最重要的特征,
而经纬度(`Latitude` / `Longitude`)的重要性也很高——这正是树模型在地理数据上的优势:
它能学会「在沿海地区,在哪个纬度区段」这种条件分支。


## 11. 残差诊断

残差 $r_i = y_i - \hat y_i$ 的分布与结构能告诉我们模型在哪些区域系统性偏差最大。
理想情况下,残差应该:
- 关于 0 对称;
- 与预测值无明显结构(无漏斗、无曲线)。


In [ ]:
best = leaderboard.iloc[0]['Model']
best_result = next(r for r in results if r['model'] == best)
best_model = best_result['fitted']

if best in ('Linear Regression', 'Ridge', 'Lasso'):
    pred = best_model.predict(X_test_scaled)
else:
    pred = best_model.predict(X_test)
residuals = y_test - pred

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(pred, y_test, alpha=0.2, s=8, color='steelblue')
lim = [min(pred.min(), y_test.min()) - 0.2, max(pred.max(), y_test.max()) + 0.2]
axes[0].plot(lim, lim, 'r--', label='Perfect prediction')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')
axes[0].set_title(f'Predicted vs True ({best})')
axes[0].legend()

axes[1].scatter(pred, residuals, alpha=0.2, s=8, color='steelblue')
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Residual (y - ŷ)')
axes[1].set_title('Residual vs Predicted')

axes[2].hist(residuals, bins=60, color='steelblue', edgecolor='white')
axes[2].axvline(0, color='red', linestyle='--')
axes[2].set_xlabel('Residual')
axes[2].set_ylabel('Count')
axes[2].set_title('Residual Distribution')

plt.tight_layout()
plt.show()

print(f'残差均值:    {residuals.mean():+.3f}')
print(f'残差标准差: {residuals.std():.3f}')


**典型现象**:
- 在 `MedHouseVal ≈ 5.0`(被截断的上限)附近,模型一致**低估**真实值——这是右截断造成的固有偏差;
- 中段(1.5–3.5)残差最对称,模型在这一段表现最好;
- 总体残差近似零均值,说明没有显著的系统性偏移。

**改进方向**(留给读者):
- 对 `MedHouseVal` 做 `log1p` 变换,缓解右偏;
- 排除 `MedHouseVal == 5.0` 的截断样本,再训练一遍;
- 工程化新特征:`AveRooms / AveBedrms`、`Population / AveOccup` 等比率特征;
- 试 `XGBoost`、`LightGBM`、`CatBoost`;
- 加入与海岸线距离的工程特征。


## 12. 深度学习对照:PyTorch MLP

到这里我们已经看过线性族、单棵树、bagging、boosting。一个自然的疑问:
**用神经网络会不会更好?**

> 💡 **预期**:在 16K+ 训练样本上,MLP 与 GBR 的差距会比 Boston(~400 样本)上**小得多**,
> 但在结构化表格数据上,**调过的梯度提升仍然倾向于占优**——这是 Kaggle 表格类竞赛多年来的共识。
> 这一节的目的:
> 1. 演示一个端到端的 PyTorch 训练流程;
> 2. 亲眼看见「数据规模 → 模型选择」的工程直觉;
> 3. 学会用早停(early stopping)+ Dropout 控制过拟合。


### 12.1 设计思路

| 组件 | 选择 | 原因 |
| --- | --- | --- |
| 输入 | 标准化后的 8 维特征 | 神经网络对量纲极敏感 |
| 隐藏层 | `8 → 64 → 64 → 32 → 1`,ReLU | 三层非线性,充分利用 16K 训练样本 |
| 正则 | Dropout(p=0.2) + 早停 + 权重衰减 | 缓解过拟合 |
| 损失 | MSE | 与其它模型一致 |
| 优化器 | Adam(lr=1e-3) | 默认稳妥选项 |
| 验证 | 从训练集再切 15% 做验证 | 严格不碰测试集 |


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch 版本: {torch.__version__}')
print(f'使用设备: {DEVICE}')


class MLPRegressor(nn.Module):
    def __init__(self, in_dim=8, hidden=(64, 64, 32), p_drop=0.2):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(p_drop)]
            prev = h
        layers += [nn.Linear(prev, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


model = MLPRegressor().to(DEVICE)
print(model)
print(f'参数量: {sum(p.numel() for p in model.parameters()):,}')


### 12.2 划分验证集 + DataLoader


In [ ]:
from sklearn.model_selection import train_test_split as tts

X_tr, X_val, y_tr, y_val = tts(X_train_scaled, y_train,
                               test_size=0.15, random_state=42)

def to_tensor(arr, dtype=torch.float32):
    return torch.tensor(arr, dtype=dtype)

train_ds = TensorDataset(to_tensor(X_tr), to_tensor(y_tr))
val_ds   = TensorDataset(to_tensor(X_val), to_tensor(y_val))

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=512)

print(f'训练: {len(train_ds)}, 验证: {len(val_ds)}, 测试: {len(X_test_scaled)}')


### 12.3 训练循环 + 早停


In [ ]:
def train_mlp(model, train_loader, val_loader,
              epochs=200, lr=1e-3, weight_decay=1e-4, patience=15):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    history = {'train_rmse': [], 'val_rmse': []}
    best_val = float('inf')
    best_state = None
    epochs_no_improve = 0

    for ep in range(1, epochs + 1):
        model.train()
        sum_sq, n = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()
            sum_sq += ((pred.detach() - yb) ** 2).sum().item()
            n += yb.numel()
        train_rmse = (sum_sq / n) ** 0.5

        model.eval()
        with torch.no_grad():
            sum_sq, n = 0.0, 0
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                pred = model(xb)
                sum_sq += ((pred - yb) ** 2).sum().item()
                n += yb.numel()
            val_rmse = (sum_sq / n) ** 0.5

        history['train_rmse'].append(train_rmse)
        history['val_rmse'].append(val_rmse)

        if val_rmse < best_val - 1e-4:
            best_val = val_rmse
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if ep % 10 == 0 or ep == 1:
            print(f'epoch {ep:3d} | train RMSE {train_rmse:.3f} | val RMSE {val_rmse:.3f}'
                  f' | best val {best_val:.3f}')

        if epochs_no_improve >= patience:
            print(f'早停于 epoch {ep}(连续 {patience} 轮无改进)')
            break

    model.load_state_dict(best_state)
    return history, best_val


model = MLPRegressor().to(DEVICE)
history, best_val_rmse = train_mlp(model, train_loader, val_loader)
print(f'\n最佳验证 RMSE: {best_val_rmse:.3f}')


### 12.4 学习曲线


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history['train_rmse'], label='Train RMSE', color='steelblue')
plt.plot(history['val_rmse'], label='Val RMSE', color='darkorange')
plt.xlabel('Epoch')
plt.ylabel('RMSE')
plt.title('MLP Training Curve')
plt.legend()
plt.tight_layout()
plt.show()


### 12.5 测试集评估 + 加入对比


In [ ]:
model.eval()
with torch.no_grad():
    pred_train = model(to_tensor(X_train_scaled).to(DEVICE)).cpu().numpy()
    pred_test  = model(to_tensor(X_test_scaled).to(DEVICE)).cpu().numpy()

train_rmse = np.sqrt(mean_squared_error(y_train, pred_train))
test_rmse  = np.sqrt(mean_squared_error(y_test, pred_test))
test_mae   = mean_absolute_error(y_test, pred_test)
test_r2    = r2_score(y_test, pred_test)

mlp_result = {
    'model': 'PyTorch MLP',
    'train_rmse': train_rmse,
    'test_rmse': test_rmse,
    'test_mae': test_mae,
    'test_r2': test_r2,
    'cv_rmse': best_val_rmse,
    'cv_std': float('nan'),
    'fitted': model,
}
results.append(mlp_result)
print_result(mlp_result)


In [ ]:
leaderboard = pd.DataFrame([
    {'Model': r['model'],
     'Train RMSE': r['train_rmse'],
     'Test RMSE':  r['test_rmse'],
     'Test MAE':   r['test_mae'],
     'Test R²':    r['test_r2']}
    for r in results
]).sort_values('Test RMSE').reset_index(drop=True)

leaderboard.round(3)


### 12.6 讨论:数据量是怎么改变结论的?

在 Boston(~400 训练样本)上,GBR 通常以巨大优势压倒 MLP。
在 California(~16K 训练样本)上,你大概率会观察到:
- MLP 的差距明显**缩小**,甚至接近随机森林;
- 但**调过的 GBR 仍然倾向于第一**。

**为什么 GBR 在表格数据上仍然占优?**
1. **inductive bias**:树模型天然适配「分段常数 + 特征交互」,这正是表格数据的结构;
2. **GBR 自带强力正则**:学习率 + 树深 + 子采样三件套等价于强正则,小到中等数据上不易过拟合;
3. **MLP 需要更多调参**:学习率、深度、宽度、正则、batch size、数据增强 ... GBR 通常 4 个超参就能调好。

**什么时候 MLP / DL 会反超?**
- 数据量到达**百万级以上**;
- 输入是图像 / 文本 / 音频 / 时序等需要表示学习的领域;
- 表格数据特别大且需要在线推理(此时 TabNet / FT-Transformer 等专用架构才有竞争力)。

> 工程直觉:**先用 GBR 起一条强基线,再判断是否需要 DL**,而不是反过来。


## 13. 总结

本 notebook 走完了一个**完整的回归任务流程**:

| 步骤 | 关键点 |
| --- | --- |
| 数据加载 | `fetch_california_housing` 一键加载,无伦理争议 |
| EDA | 看分布、看缺失、看异常(右截断 \$500K) |
| 地理可视化 | 用经纬度散点直接画出加州房价地图 |
| 相关性 | `MedInc` 一枝独秀,`AveRooms` / `AveBedrms` 共线 |
| 预处理 | 仅在训练集 fit scaler,避免泄漏 |
| 基线 | 线性回归是不可省略的对照 |
| 正则化 | Ridge 抗共线性,Lasso 自动选特征 |
| 树模型 | 不需标准化、自然捕捉非线性与地理交互 |
| 调参 | GridSearchCV + 交叉验证 |
| 解释 | 系数 / 特征重要性 / 残差三件套 |
| 深度学习 | PyTorch MLP + 早停,作为对照而非首选 |

### 学到的核心思想
1. **永远先有一条基线**——没有基线,所有「提升」都没有意义;
2. **标准化只在线性族 / 神经网络上需要**,树模型不必;
3. **超参搜索必须在训练集内做交叉验证**,绝不碰测试集;
4. **残差诊断 ≠ 评估指标**——它告诉你「错在哪里」,比单一 RMSE 更有价值;
5. **特征重要性 ≠ 因果**——它只反映模型的依赖,不是真实世界的因果关系;
6. **数据规模决定模型选择**——小到中等表格数据上 GBR 通常 ≥ NN,先 GBR 强基线、再考虑 DL。

### 进一步阅读
- *An Introduction to Statistical Learning* (ISLR) — 第 3、6、8 章
- scikit-learn 官方文档:`Pipeline`, `ColumnTransformer`, `cross_validate`, `HistGradientBoostingRegressor`
- 用 SHAP 值做更严谨的可解释性分析
- 表格数据上的现代 DL 架构:TabNet、FT-Transformer、SAINT
- 加州房价的扩展数据集:[Inside Airbnb](http://insideairbnb.com/) 等更新源
